# Summary Combo — Datewise, Precise and Combined workbooks

Builds three workbooks from the five sources (brands, crates, Mehul returns, Hansa returns, sales revenue):

| Workbook | Content |
|---|---|
| `Datewise_Workbook.xlsx` | 5 sheets, every date kept (one row per date × customer × product × type) |
| `Precise_Workbook.xlsx` | 5 sheets, no dates: one summarised row per customer (× product × type) |
| `Combined_Workbook.xlsx` | 1 sheet, everything together: one `RECORD_TYPE` column says which source/type a row is, one `QTY` and one `AMOUNT` column |

Sheets in the first two workbooks: `crates`, `mehul_returns`, `hansa_returns`, `sales_revenue`, `brands_perfomance`.

**Conventions**
- Every sheet starts with all the customer-info columns, exactly as they are in the customer-info file, matched on the customer code / A/C. `CUSTOMER_MATCHED` = `No` flags a code that is not in the customer-info file.
- Everything else is unpivoted so each sheet has few columns: products go into a `PRODUCT` column, invoice/credit and fresh/expired go into a `TYPE` column, and there is one quantity column per sheet (named after its source, e.g. `BRANDS_QTY`). Rows whose quantity is zero are dropped.
- Credit / return quantities are positive numbers; the `TYPE` column tells you they are credits.
- Brands and crates: an invoice that is exactly reversed by a credit note is removed (as in `BRANDS.ipynb` / `CRATES.ipynb`). The reversing credit is removed with it, so it is not counted twice (set `KEEP_CANCELLED_CREDITS = True` to keep it).

## Config

In [43]:
import re, calendar
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

DATA = Path("data")
CUSTOMER_INFO_FILE = DATA / "customer-info" / "customer-info.xlsx"
BRANDS_FILE        = DATA / "brands" / "oct-week3-brand.xlsx"          # sheets: Brands-inv, Brands-cn
CRATES_FILE        = DATA / "crates" / "oct-week3-crates.xlsx"         # sheets: Crates-inv, Crates-cn
MEHUL_FILE         = DATA / "mehul-returns" / "OCTOBER_RETURNS.xlsx"   # one sheet per day: 1ST, 2ND, ...
HANSA_FILE         = DATA / "hansa-returns" / "oct-week3-hansa-returns.xlsx"
SALES_REV_FILE     = DATA / "sales-revenue" / "october-week3-sales-revenue.xlsx"  # sheets: Invoices, Credits

OUTPUT_DIR = Path("report") / "test"

# Mehul's daily sheets only carry the day (1ST, 2ND ...), so the month has to be given here
MEHUL_YEAR, MEHUL_MONTH = 2026, 10
# Extra Mehul sheets that are NOT named 1ST/2ND/... but should be read, e.g. {"Sheet14": 3} = treat as 3rd of the month
MEHUL_SHEET_DATE_OVERRIDES = {}

# False = an invoice reversed by an identical credit note disappears on both sides
KEEP_CANCELLED_CREDITS = False

# Brands: columns that must match (credit = -invoice) for a credit note to cancel an invoice (same list as BRANDS.ipynb)
BRAND_MATCH_COLS = ['P', 'S', 'B', '200g', 'O.P.', '800g', '600g', 'MANDAZI', 'DOUGHNUTS',
                    'MANDO FAMILY PACK', '4 Square', 'XTRA LOONG', 'B-BITEZ', 'QTY']

## Helpers — product names, customer codes, customer info

In [44]:
def key(x):
    return re.sub(r"[^A-Z0-9]", "", str(x).upper())

# One spelling per product, whatever the source file calls it (200g/200G, O.P/O.P., BBITEZ/B-BITEZ ...)
PRODUCTS = ['P', 'S', 'B', '200g', 'O.P.', '800g', '1500g', 'ST', '600g', 'KATIKATI', 'MANDAZI', 'DOUGHNUTS',
            '50-50 SLICED', 'MANDO FAMILY PACK', '4 Square', 'XTRA LOONG', 'B-BITEZ']
_PRODUCT_LOOKUP = {key(p): p for p in PRODUCTS}
_PRODUCT_LOOKUP.update({'4SQ': '4 Square', 'XLOONG': 'XTRA LOONG', 'FAMILYPACK': 'MANDO FAMILY PACK',
                        'BB': 'B-BITEZ', '5050': '50-50 SLICED'})
# Names not listed above (e.g. Mehul's MANDO, MAICHO) are kept as written (upper-case). Add aliases above to merge them.
def canon(x):
    return _PRODUCT_LOOKUP.get(key(x), str(x).strip().upper())

META_KEYS = {'DATE': 'DATE', 'DATES': 'DATE', 'AC': 'A/C', 'CUSTOMER': 'NAME', 'NAME': 'NAME', 'QTY': 'QTY',
             'NO': 'NO', 'INVOICENO': 'INVOICE_NO', 'INVOICE': 'INVOICE_NO', 'TYPE': 'TYPE', 'STATUS': 'STATUS'}
META_COLS = set(META_KEYS.values())

def std_cols(df):
    df = df.copy()
    df.columns = [META_KEYS.get(key(c)) or canon(c) for c in df.columns]
    return df

def to_code(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

def code_key(s):
    """Code as a whole number; a missing/invalid code becomes 0 so those rows are not silently dropped by groupby."""
    return to_code(s).fillna(0).astype("int64")

def as_qty(s):
    """Whole numbers stay int64; if a source ever holds decimals they are kept, never truncated."""
    s = pd.to_numeric(s).fillna(0)
    return s.round().astype("int64") if np.allclose(s, s.round()) else s.astype("float64")

def to_date(s):
    return pd.to_datetime(s, errors="coerce").dt.normalize()

# ---- customer info
cust = pd.read_excel(CUSTOMER_INFO_FILE)
cust["CODE"] = to_code(cust["CODE"])
cust = cust.dropna(subset=["CODE"])
_dup = cust["CODE"].duplicated()
if _dup.any():
    print(f"WARNING: {_dup.sum()} duplicate CODE rows in customer info - first one kept: {sorted(cust.loc[_dup, 'CODE'].tolist())}")
cust = cust[~_dup].reset_index(drop=True)
INFO_COLS = ["CODE"] + [c for c in cust.columns if c != "CODE"]
cust = cust[INFO_COLS]
ID_COLS = INFO_COLS + ["CUSTOMER_MATCHED"]
print("Customer info:", cust.shape, INFO_COLS)

def first_name_by_code(df, code_col="A/C", name_col="NAME"):
    t = df[[code_col, name_col]].dropna()
    t = t.assign(_c=to_code(t[code_col])).dropna(subset=["_c"])
    return t.drop_duplicates("_c").set_index("_c")[name_col].astype(str).to_dict()

def attach_customer(df, code_col="CODE", name_col=None, fallback=None, label=""):
    """Add every customer-info column by code. Codes missing from customer info keep a source name (if any) and are flagged."""
    df = df.copy()
    n_in = len(df)
    c = to_code(df[code_col])
    df["CODE"] = c.mask(c == 0)                      # 0 = "no code"
    if code_col != "CODE":
        df = df.drop(columns=code_col)
    src = pd.Series("", index=df.index, dtype="object")
    if fallback is not None:
        src = df["CODE"].map(lambda c: fallback.get(int(c), "") if pd.notna(c) else "").astype("object")
    if name_col:
        nm = df[name_col].fillna("").astype(str).str.strip().astype("object")
        src = nm.where(nm != "", src)
        df = df.drop(columns=name_col)
    out = df.merge(cust.assign(CUSTOMER_MATCHED="Yes"), on="CODE", how="left")
    assert len(out) == n_in, "customer merge changed the row count"
    out["CUSTOMER_MATCHED"] = out["CUSTOMER_MATCHED"].fillna("No")
    miss = (out["CUSTOMER_MATCHED"] == "No").to_numpy()
    out["CUSTOMER NAME"] = out["CUSTOMER NAME"].astype("object")
    out.loc[miss, "CUSTOMER NAME"] = src.to_numpy()[miss]
    n_bad = int(miss.sum())
    if n_bad:
        bad = out.loc[miss, "CODE"]
        print(f"  {label}: {n_bad} rows ({bad.nunique(dropna=False)} codes) not in customer info, e.g. {bad.dropna().unique()[:10].tolist()}")
    return out

Customer info: (1221, 6) ['CODE', 'CUSTOMER NAME', 'CATEGORY', 'ROUTE NAME', 'SALES_REP', 'MERCHANDISERS']


## Helper — cancel invoices that a credit note exactly reverses

In [45]:
def cancel_matched(inv, cn, exact, negated):
    """
    One credit note cancels ONE invoice (not every duplicate).
    exact   : columns that must be equal on both sides
    negated : columns where credit == -invoice
    Returns invoices kept, credits kept (not matched), and the matched credits.
    """
    inv = inv.reset_index(drop=True)
    cn = cn.reset_index(drop=True)
    keys = exact + negated
    ik = inv[keys].copy()
    ck = cn[keys].copy()
    for c in negated:
        ck[c] = -ck[c]
    ik["_n"] = ik.groupby(keys, dropna=False).cumcount()
    ck["_n"] = ck.groupby(keys, dropna=False).cumcount()
    k2 = keys + ["_n"]
    inv_hit = ik.merge(ck[k2].assign(_h=1), on=k2, how="left")["_h"].notna().to_numpy()
    cn_hit = ck.merge(ik[k2].assign(_h=1), on=k2, how="left")["_h"].notna().to_numpy()
    return inv[~inv_hit].copy(), cn[~cn_hit].copy(), cn[cn_hit].copy()

# Brands

In [46]:
print("BRANDS")
b_inv = std_cols(pd.read_excel(BRANDS_FILE, sheet_name="Brands-inv"))
b_cn  = std_cols(pd.read_excel(BRANDS_FILE, sheet_name="Brands-cn"))
b_prods = [c for c in b_inv.columns if c not in META_COLS]
for d in (b_inv, b_cn):
    d["DATE"] = to_date(d["DATE"])
    d["A/C"] = code_key(d["A/C"])
    for c in b_prods + ["QTY"]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

match_cols = [c for c in BRAND_MATCH_COLS if c in b_inv.columns and c in b_cn.columns]
b_inv_kept, b_cn_open, b_cn_cancelled = cancel_matched(b_inv, b_cn, exact=["DATE", "A/C"], negated=match_cols)
print(f"  invoices {len(b_inv)} -> {len(b_inv_kept)} after removing {len(b_inv) - len(b_inv_kept)} reversed by a credit note")
print(f"  credit notes {len(b_cn)}: {len(b_cn_cancelled)} cancelled an invoice, {len(b_cn_open)} stand alone")
b_cn_use = b_cn if KEEP_CANCELLED_CREDITS else b_cn_open

def brand_long(df, value_name):
    g = df.groupby(["DATE", "A/C"], as_index=False)[b_prods].sum()
    long = g.melt(id_vars=["DATE", "A/C"], var_name="PRODUCT", value_name=value_name)
    return long[long[value_name] != 0]

_cn_pos = b_cn_use.copy()
_cn_pos[b_prods] = _cn_pos[b_prods].abs()
brands_dw = (pd.concat([brand_long(b_inv_kept, "BRANDS_INV_QTY"), brand_long(_cn_pos, "BRANDS_CN_QTY")])
             .groupby(["DATE", "A/C", "PRODUCT"], as_index=False).sum())
brands_dw = attach_customer(brands_dw, "A/C", fallback=first_name_by_code(pd.concat([b_inv, b_cn])), label="brands")
BRANDS_M = ["BRANDS_INV_QTY", "BRANDS_CN_QTY"]
brands_dw[BRANDS_M] = brands_dw[BRANDS_M].fillna(0).apply(as_qty)
brands_dw = brands_dw[["DATE"] + ID_COLS + ["PRODUCT"] + BRANDS_M].sort_values(["DATE", "CODE", "PRODUCT"]).reset_index(drop=True)
brands_dw.head()

BRANDS
  invoices 7023 -> 6861 after removing 162 reversed by a credit note
  credit notes 3045: 162 cancelled an invoice, 2883 stand alone


,DATE,CODE,CUSTOMER NAME,CATEGORY,ROUTE NAME,SALES_REP,MERCHANDISERS,CUSTOMER_MATCHED,PRODUCT,BRANDS_INV_QTY,BRANDS_CN_QTY
0,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,200g,192,0
1,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,600g,28,0
2,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,B,120,0
3,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,O.P.,45,0
4,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,S,315,0


# Crates

In [47]:
print("CRATES")
c_inv = std_cols(pd.read_excel(CRATES_FILE, sheet_name="Crates-inv"))
c_cn  = std_cols(pd.read_excel(CRATES_FILE, sheet_name="Crates-cn"))
c_cn = c_cn[c_cn["STATUS"].astype(str).str.strip().str.upper() == "CN"]      # as in CRATES.ipynb
for d in (c_inv, c_cn):
    d["DATE"] = to_date(d["DATE"])
    d["A/C"] = code_key(d["A/C"])
    d["QTY"] = pd.to_numeric(d["QTY"], errors="coerce").fillna(0)

# CRATES.ipynb matches on A/C + QTY only (a credit can reverse an earlier day's invoice)
c_inv_kept, c_cn_open, c_cn_cancelled = cancel_matched(c_inv, c_cn, exact=["A/C", "QTY"], negated=[])
print(f"  invoices {len(c_inv)} -> {len(c_inv_kept)} after removing {len(c_inv) - len(c_inv_kept)} reversed by a credit note")
print(f"  credit notes {len(c_cn)}: {len(c_cn_cancelled)} cancelled an invoice, {len(c_cn_open)} stand alone")
c_cn_use = c_cn if KEEP_CANCELLED_CREDITS else c_cn_open

crates_dw = (pd.concat([
        c_inv_kept.groupby(["DATE", "A/C"], as_index=False)["QTY"].sum().rename(columns={"QTY": "CRATES_INV_QTY"}),
        c_cn_use.groupby(["DATE", "A/C"], as_index=False)["QTY"].sum().rename(columns={"QTY": "CRATES_CN_QTY"})])
    .groupby(["DATE", "A/C"], as_index=False).sum())
crates_dw = attach_customer(crates_dw, "A/C", label="crates")
CRATES_M = ["CRATES_INV_QTY", "CRATES_CN_QTY"]
crates_dw[CRATES_M] = crates_dw[CRATES_M].fillna(0).apply(as_qty)
crates_dw = crates_dw[["DATE"] + ID_COLS + CRATES_M].sort_values(["DATE", "CODE"]).reset_index(drop=True)
crates_dw.head()

CRATES
  invoices 7029 -> 6179 after removing 850 reversed by a credit note
  credit notes 3047: 850 cancelled an invoice, 2197 stand alone


,DATE,CODE,CUSTOMER NAME,CATEGORY,ROUTE NAME,SALES_REP,MERCHANDISERS,CUSTOMER_MATCHED,CRATES_INV_QTY,CRATES_CN_QTY
0,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,47,0
1,2026-10-11,57,KARATINA,DIST,KARATINA ROUTE,Lillian,Unkown,Yes,90,0
2,2026-10-11,62,RUIRU,DIST,RUIRU ROUTE,Magdaline,Unkown,Yes,134,0
3,2026-10-11,63,MACHAKOS,DIST,MACHAKOS ROUTE,Saidi,Unkown,Yes,96,0
4,2026-10-11,82,KANGEMI,DIST,KANGEMI ROUTE,Dorothy,Unkown,Yes,42,0


# Mehul returns
The month file has one sheet per day (`1ST`, `2ND`, ...). Only those sheets are read, and the day in the sheet name becomes the date.
Other tabs (`Sheet14`, `CommentLists`, `APRIL RANGE`, ...) are skipped and listed; if one of them is a real day, add it to `MEHUL_SHEET_DATE_OVERRIDES`.
Rows with no code get it from the `(1234)` in the name, or from an exact name match in customer info.

In [48]:
print("MEHUL RETURNS")
ORD = re.compile(r"^\s*(\d{1,2})\s*(ST|ND|RD|TH)(\w*)\s*$", re.I)
xl = pd.ExcelFile(MEHUL_FILE)
days_in_month = calendar.monthrange(MEHUL_YEAR, MEHUL_MONTH)[1]

by_day = {}
for s in xl.sheet_names:
    m = ORD.match(s)
    if m:
        by_day.setdefault(int(m.group(1)), []).append((m.group(3) != "", s))   # (misspelt?, sheet)
chosen, skipped = [], []
for day, lst in sorted(by_day.items()):
    lst.sort(key=lambda t: t[0])             # properly spelt name first
    if day > days_in_month:
        skipped += [s for _, s in lst]; continue
    chosen.append((day, lst[0][1])); skipped += [s for _, s in lst[1:]]
for s, day in MEHUL_SHEET_DATE_OVERRIDES.items():
    chosen.append((day, s)); skipped = [x for x in skipped if x != s]
skipped += [s for s in xl.sheet_names if s not in {c[1] for c in chosen} and s not in skipped]
print(f"  reading {len(chosen)} daily sheets: {[s for _, s in chosen]}")
print(f"  skipped sheets: {skipped}")

def parse_mehul_sheet(sheet):
    raw = xl.parse(sheet, header=None)
    if raw.shape[0] < 4 or raw.shape[1] < 6:
        return None
    brands, types = raw.iloc[1, 5:].tolist(), raw.iloc[2, 5:].tolist()
    cur, seen, spec = None, set(), []
    for b, t in zip(brands, types):
        if pd.notna(b) and str(b).strip():
            cur = str(b).strip().upper()
        tt = str(t).strip().upper() if pd.notna(t) else ""
        tt = {"EX": "EXP", "1": "EXP", "1.0": "EXP"}.get(tt, tt)
        if cur and cur != "TOTAL" and tt in ("FRESH", "EXP") and (canon(cur), tt) not in seen:
            seen.add((canon(cur), tt)); spec.append((canon(cur), tt))
        else:
            spec.append(None)                # TOTAL block, comments, repeated brand blocks (first one wins)
    body = raw.iloc[3:].reset_index(drop=True)
    out = pd.DataFrame({"CODE": body[3], "NAME": body[4]})
    for pos, sp in enumerate(spec, start=5):
        if sp is not None:
            out[f"{sp[0]}|{sp[1]}"] = pd.to_numeric(body[pos], errors="coerce").fillna(0)
    nm = out["NAME"].fillna("").astype(str).str.strip()
    out = out[~((out["CODE"].isna()) & (nm == ""))]
    out = out[nm.loc[out.index].str.upper() != "TOTAL"]
    return out

frames = []
for day, s in chosen:
    f = parse_mehul_sheet(s)
    if f is not None and len(f):
        f["DATE"] = pd.Timestamp(MEHUL_YEAR, MEHUL_MONTH, day)
        frames.append(f)
m_raw = pd.concat(frames, ignore_index=True)
val_cols = [c for c in m_raw.columns if "|" in c]
m_raw[val_cols] = m_raw[val_cols].fillna(0)

# ---- find missing codes
def norm_name(s):
    s = re.sub(r"\(\s*\d+\s*\)\s*$", "", str(s).upper())
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]+", " ", s)).strip()
_name_to_code = (cust.assign(_k=cust["CUSTOMER NAME"].astype(str).map(norm_name))
                     .drop_duplicates("_k", keep=False).set_index("_k")["CODE"].to_dict())
nm = m_raw["NAME"].fillna("").astype(str)
code0 = to_code(m_raw["CODE"])
n0 = int(code0.isna().sum())
code1 = code0.fillna(to_code(nm.str.extract(r"\((\d+)\)\s*$", expand=False)))
code2 = code1.fillna(to_code(nm.map(lambda s: _name_to_code.get(norm_name(s), np.nan))))
print(f"  rows without a code: {n0} -> {int(code2.isna().sum())} after inferring from the name")
m_raw["CODE"] = code2
_fallback = first_name_by_code(m_raw, "CODE", "NAME")
m_raw["NAME"] = np.where(m_raw["CODE"].notna(), "", nm)         # name only matters for rows that still have no code
m_raw["CODE"] = m_raw["CODE"].fillna(0).astype("int64")          # 0 = "no code" so uncoded rows are not dropped by groupby

# ---- pivot brands into rows, FRESH / EXP into two columns
long = m_raw.melt(id_vars=["DATE", "CODE", "NAME"], value_vars=val_cols, var_name="k", value_name="v")
long[["PRODUCT", "TYPE"]] = long["k"].str.split("|", expand=True)
mehul_dw = (long.groupby(["DATE", "CODE", "NAME", "PRODUCT", "TYPE"])["v"].sum().unstack("TYPE", fill_value=0).reset_index())
for t in ("FRESH", "EXP"):
    if t not in mehul_dw.columns:
        mehul_dw[t] = 0
mehul_dw = mehul_dw.rename(columns={"FRESH": "MEHUL_FRESH_RTN_QTY", "EXP": "MEHUL_EXP_RTN_QTY"})
mehul_dw["MEHUL_TOTAL_RTN_QTY"] = mehul_dw["MEHUL_FRESH_RTN_QTY"] + mehul_dw["MEHUL_EXP_RTN_QTY"]
mehul_dw = mehul_dw[mehul_dw["MEHUL_TOTAL_RTN_QTY"] != 0].copy()
MEHUL_M = ["MEHUL_FRESH_RTN_QTY", "MEHUL_EXP_RTN_QTY", "MEHUL_TOTAL_RTN_QTY"]
mehul_dw = attach_customer(mehul_dw, "CODE", name_col="NAME", fallback=_fallback, label="mehul")
mehul_dw[MEHUL_M] = mehul_dw[MEHUL_M].apply(as_qty)
mehul_dw = mehul_dw[["DATE"] + ID_COLS + ["PRODUCT"] + MEHUL_M].sort_values(["DATE", "CODE", "PRODUCT"]).reset_index(drop=True)
mehul_dw.head()

MEHUL RETURNS
  reading 7 daily sheets: ['11TH', '12TH', '13TH', '14TH', '15TH', '16TH', '17TH']
  skipped sheets: ['12thh', 'Sheet14', 'Sheet15', 'Sheet16', 'Sheet17', 'Sheet18', 'Sheet12', 'Sheet23', 'Sheet24', 'Sheet6', 'Sheet5', 'CommentLists', 'Sheet25', 'Sheet26', 'Sheet27', 'Sheet28', 'Sheet19', 'Sheet13', 'Sheet11', 'Sheet7', 'Sheet8', 'Sheet9', 'APRIL RANGE', 'MARCH RANGE ', 'ELDORET', 'NAKURU', 'Sheet10', 'Sheet1', 'Sheet2', 'Sheet4', 'ALL CUSTOMERS & ROUTES', 'DISTRIBUTORS', 'Sheet3']
  rows without a code: 91 -> 91 after inferring from the name
  mehul: 400 rows (1 codes) not in customer info, e.g. []


,DATE,CODE,CUSTOMER NAME,CATEGORY,ROUTE NAME,SALES_REP,MERCHANDISERS,CUSTOMER_MATCHED,PRODUCT,MEHUL_FRESH_RTN_QTY,MEHUL_EXP_RTN_QTY,MEHUL_TOTAL_RTN_QTY
0,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,600g,0,1,1
1,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,S,0,2,2
2,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,XTRA LOONG,0,1,1
3,2026-10-11,57,KARATINA,DIST,KARATINA ROUTE,Lillian,Unkown,Yes,600g,1,0,1
4,2026-10-11,57,KARATINA,DIST,KARATINA ROUTE,Lillian,Unkown,Yes,B,4,0,4


# Hansa returns

In [49]:
print("HANSA RETURNS")
h = std_cols(pd.read_excel(HANSA_FILE))          # first sheet; add sheet_name=... if the file has several
h_prods = [c for c in h.columns if c not in META_COLS]
h["DATE"] = to_date(h["DATE"])
h["A/C"] = code_key(h["A/C"])
for c in h_prods:
    h[c] = pd.to_numeric(h[c], errors="coerce").fillna(0)
hg = h.groupby(["DATE", "A/C"], as_index=False)[h_prods].sum()
hansa_dw = hg.melt(id_vars=["DATE", "A/C"], var_name="PRODUCT", value_name="HANSA_RTN_QTY")
hansa_dw = hansa_dw[hansa_dw["HANSA_RTN_QTY"] != 0]
hansa_dw = attach_customer(hansa_dw, "A/C", fallback=first_name_by_code(h), label="hansa")
HANSA_M = ["HANSA_RTN_QTY"]
hansa_dw[HANSA_M] = hansa_dw[HANSA_M].apply(as_qty)
hansa_dw = hansa_dw[["DATE"] + ID_COLS + ["PRODUCT"] + HANSA_M].sort_values(["DATE", "CODE", "PRODUCT"]).reset_index(drop=True)
print(f"  products: {h_prods}")
hansa_dw.head()

HANSA RETURNS
  products: ['P', 'S', 'B', '200g', 'O.P.', '800g', '1500g', 'ST', '600g', 'KATIKATI', 'MANDAZI', 'DOUGHNUTS', '50-50 SLICED', 'MANDO FAMILY PACK', '4 Square', 'XTRA LOONG', 'B-BITEZ']


,DATE,CODE,CUSTOMER NAME,CATEGORY,ROUTE NAME,SALES_REP,MERCHANDISERS,CUSTOMER_MATCHED,PRODUCT,HANSA_RTN_QTY
0,2026-10-11,6372,MAUA SALES - II,VAN,MAUA 2 SALES,Unkown,Unkown,Yes,200g,6
1,2026-10-11,6372,MAUA SALES - II,VAN,MAUA 2 SALES,Unkown,Unkown,Yes,600g,3
2,2026-10-11,6372,MAUA SALES - II,VAN,MAUA 2 SALES,Unkown,Unkown,Yes,B,1
3,2026-10-11,6372,MAUA SALES - II,VAN,MAUA 2 SALES,Unkown,Unkown,Yes,O.P.,4
4,2026-10-11,6372,MAUA SALES - II,VAN,MAUA 2 SALES,Unkown,Unkown,Yes,S,3


# Sales revenue
Invoices and Credits are read from their own sheets (the old cell read the first sheet twice). Credit qty/amount are made positive.

In [50]:
print("SALES REVENUE")
def clean_rev(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("(", "", regex=False)
                  .str.replace(")", "", regex=False).str.replace(".", "", regex=False))
    df["DATE"] = to_date(df["date"])
    df["A/C"] = code_key(df["a/c_no"])
    for c in ("qty", "amount"):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df

rev_inv = clean_rev(pd.read_excel(SALES_REV_FILE, sheet_name="Invoices"))
rev_cn  = clean_rev(pd.read_excel(SALES_REV_FILE, sheet_name="Credits"))
rev_cn[["qty", "amount"]] = rev_cn[["qty", "amount"]].abs()

def rev_group(df, qty_name, amt_name):
    return (df.groupby(["DATE", "A/C"], as_index=False)[["qty", "amount"]].sum()
              .rename(columns={"qty": qty_name, "amount": amt_name}))
rev_dw = pd.concat([rev_group(rev_inv, "REV_INV_QTY", "REV_INV_AMT"),
                    rev_group(rev_cn, "REV_CN_QTY", "REV_CN_AMT")]).groupby(["DATE", "A/C"], as_index=False).sum()
_names = first_name_by_code(pd.concat([rev_inv, rev_cn]).rename(columns={"customers": "NAME"}))
rev_dw = attach_customer(rev_dw, "A/C", fallback=_names, label="revenue")
REV_M = ["REV_INV_QTY", "REV_INV_AMT", "REV_CN_QTY", "REV_CN_AMT"]
rev_dw[REV_M] = rev_dw[REV_M].fillna(0)
rev_dw[["REV_INV_QTY", "REV_CN_QTY"]] = rev_dw[["REV_INV_QTY", "REV_CN_QTY"]].apply(as_qty)
rev_dw = rev_dw[["DATE"] + ID_COLS + REV_M].sort_values(["DATE", "CODE"]).reset_index(drop=True)
rev_dw.head()

SALES REVENUE


,DATE,CODE,CUSTOMER NAME,CATEGORY,ROUTE NAME,SALES_REP,MERCHANDISERS,CUSTOMER_MATCHED,REV_INV_QTY,REV_INV_AMT,REV_CN_QTY,REV_CN_AMT
0,2026-10-11,55,RUIRUNGU,DIST,KAMAKWA ROUTE,Lillian,Unkown,Yes,721,34548.0,0,0.0
1,2026-10-11,57,KARATINA,DIST,KARATINA ROUTE,Lillian,Unkown,Yes,1356,68098.5,0,0.0
2,2026-10-11,62,RUIRU,DIST,RUIRU ROUTE,Magdaline,Unkown,Yes,1698,90675.0,0,0.0
3,2026-10-11,63,MACHAKOS,DIST,MACHAKOS ROUTE,Saidi,Unkown,Yes,1442,71587.5,0,0.0
4,2026-10-11,82,KANGEMI,DIST,KANGEMI ROUTE,Dorothy,Unkown,Yes,696,31503.0,0,0.0


# Reshape to few columns
Wide tables above are only working tables. The sheets are built from them by unpivoting everything except the customer info.

In [51]:
def unpivot_qty(df, metric_map, qty_name, extra=()):
    """metric columns -> one TYPE column + one quantity column."""
    ids = ["DATE"] + ID_COLS + list(extra)
    l = df.melt(id_vars=ids, value_vars=list(metric_map), var_name="TYPE", value_name=qty_name)
    l["TYPE"] = l["TYPE"].map(metric_map)
    l = l[l[qty_name] != 0]
    l[qty_name] = as_qty(l[qty_name])
    return l[ids + ["TYPE", qty_name]].sort_values(["DATE", "CODE", *extra, "TYPE"]).reset_index(drop=True)

brands_sheet = unpivot_qty(brands_dw, {"BRANDS_INV_QTY": "INVOICE", "BRANDS_CN_QTY": "CREDIT"}, "BRANDS_QTY", ["PRODUCT"])
crates_sheet = unpivot_qty(crates_dw, {"CRATES_INV_QTY": "INVOICE", "CRATES_CN_QTY": "CREDIT"}, "CRATES_QTY")
mehul_sheet  = unpivot_qty(mehul_dw,  {"MEHUL_FRESH_RTN_QTY": "FRESH", "MEHUL_EXP_RTN_QTY": "EXPIRED"}, "MEHUL_RTN_QTY", ["PRODUCT"])
hansa_sheet  = hansa_dw                                      # already DATE, customer info, PRODUCT, HANSA_RTN_QTY

_ri = rev_dw[["DATE"] + ID_COLS + ["REV_INV_QTY", "REV_INV_AMT"]].set_axis(["DATE"] + ID_COLS + ["REV_QTY", "REV_AMT"], axis=1).assign(TYPE="INVOICE")
_rc = rev_dw[["DATE"] + ID_COLS + ["REV_CN_QTY", "REV_CN_AMT"]].set_axis(["DATE"] + ID_COLS + ["REV_QTY", "REV_AMT"], axis=1).assign(TYPE="CREDIT")
rev_sheet = pd.concat([_ri, _rc])
rev_sheet = rev_sheet[(rev_sheet["REV_QTY"] != 0) | (rev_sheet["REV_AMT"] != 0)]
rev_sheet = (rev_sheet[["DATE"] + ID_COLS + ["TYPE", "REV_QTY", "REV_AMT"]]
             .sort_values(["DATE", "CODE", "TYPE"]).reset_index(drop=True))

datewise = {
    "crates":            crates_sheet,
    "mehul_returns":     mehul_sheet,
    "hansa_returns":     hansa_sheet,
    "sales_revenue":     rev_sheet,
    "brands_perfomance": brands_sheet,
}
for k, v in datewise.items():
    print(f"{k}: {len(v):,} rows, {v.shape[1]} columns")

crates: 5,725 rows, 10 columns
mehul_returns: 6,069 rows, 11 columns
hansa_returns: 4,254 rows, 10 columns
sales_revenue: 6,825 rows, 11 columns
brands_perfomance: 31,359 rows, 11 columns


# Precise workbook
The datewise sheets summed per customer (× product × type). `PERIOD` shows the dates each source covers (the sources do not all cover the same dates).
In `sales_revenue`, `PCT_RETURN` = credit amount / invoice amount for the customer, shown on its `CREDIT` row.

In [52]:
def precise(df, extra_keys, qty_cols):
    by = ID_COLS + list(extra_keys)
    g = (df.groupby(by, dropna=False)
           .agg(**{c: (c, "sum") for c in qty_cols}, _from=("DATE", "min"), _to=("DATE", "max"))
           .reset_index())
    a, b = g["_from"].dt.strftime("%Y-%m-%d"), g["_to"].dt.strftime("%Y-%m-%d")
    g["PERIOD"] = np.where(a == b, a, a + " to " + b)
    return g[by + ["PERIOD"] + qty_cols]

def precise_typed(df, extra_keys, qty_col):
    return (df.groupby(ID_COLS + list(extra_keys) + ["TYPE"], dropna=False)
              .agg(**{qty_col: (qty_col, "sum")}, _from=("DATE", "min"), _to=("DATE", "max"))
              .reset_index()
              .pipe(lambda g: g.assign(PERIOD=np.where(g["_from"] == g["_to"], g["_from"].dt.strftime("%Y-%m-%d"),
                                                      g["_from"].dt.strftime("%Y-%m-%d") + " to " + g["_to"].dt.strftime("%Y-%m-%d"))))
              [ID_COLS + list(extra_keys) + ["TYPE", "PERIOD", qty_col]]
              .sort_values(ID_COLS[:1] + list(extra_keys) + ["TYPE"]).reset_index(drop=True))

# revenue: both INVOICE and CREDIT row for every customer (zeros kept) so PCT_RETURN always has a home
_rev_tot = (rev_dw.groupby(ID_COLS, dropna=False)
                  .agg(INVOICE_QTY=("REV_INV_QTY", "sum"), INVOICE_AMT=("REV_INV_AMT", "sum"),
                       CREDIT_QTY=("REV_CN_QTY", "sum"), CREDIT_AMT=("REV_CN_AMT", "sum"),
                       _from=("DATE", "min"), _to=("DATE", "max")).reset_index())
_rev_tot["PERIOD"] = np.where(_rev_tot["_from"] == _rev_tot["_to"], _rev_tot["_from"].dt.strftime("%Y-%m-%d"),
                              _rev_tot["_from"].dt.strftime("%Y-%m-%d") + " to " + _rev_tot["_to"].dt.strftime("%Y-%m-%d"))
_pct = np.where(_rev_tot["INVOICE_AMT"] != 0, _rev_tot["CREDIT_AMT"] / _rev_tot["INVOICE_AMT"], 0).round(4)
_pi = _rev_tot[ID_COLS + ["PERIOD", "INVOICE_QTY", "INVOICE_AMT"]].set_axis(ID_COLS + ["PERIOD", "REV_QTY", "REV_AMT"], axis=1).assign(TYPE="INVOICE", PCT_RETURN=np.nan)
_pc = _rev_tot[ID_COLS + ["PERIOD", "CREDIT_QTY", "CREDIT_AMT"]].set_axis(ID_COLS + ["PERIOD", "REV_QTY", "REV_AMT"], axis=1).assign(TYPE="CREDIT", PCT_RETURN=_pct)
rev_precise = (pd.concat([_pi, _pc])[ID_COLS + ["TYPE", "PERIOD", "REV_QTY", "REV_AMT", "PCT_RETURN"]]
               .sort_values(["CODE", "TYPE"]).reset_index(drop=True))

precise_sheets = {
    "crates":            precise_typed(crates_sheet, [], "CRATES_QTY"),
    "mehul_returns":     precise_typed(mehul_sheet, ["PRODUCT"], "MEHUL_RTN_QTY"),
    "hansa_returns":     precise(hansa_sheet, ["PRODUCT"], ["HANSA_RTN_QTY"]).sort_values(["CODE", "PRODUCT"]).reset_index(drop=True),
    "sales_revenue":     rev_precise,
    "brands_perfomance": precise_typed(brands_sheet, ["PRODUCT"], "BRANDS_QTY"),
}
for k, v in precise_sheets.items():
    print(f"{k}: {len(v):,} rows, {v.shape[1]} columns")

crates: 1,157 rows, 10 columns
mehul_returns: 2,749 rows, 11 columns
hansa_returns: 2,176 rows, 10 columns
sales_revenue: 1,790 rows, 12 columns
brands_perfomance: 7,835 rows, 11 columns


# Combined workbook
One sheet, one row per date × customer × product × `RECORD_TYPE`.
`RECORD_TYPE` names the source and what the row is (`BRANDS_INVOICE`, `BRANDS_CREDIT`, `CRATES_INVOICE`, `CRATES_CREDIT`, `HANSA_RETURN`, `MEHUL_FRESH_RETURN`, `MEHUL_EXPIRED_RETURN`, `REVENUE_INVOICE`, `REVENUE_CREDIT`).
Crates and revenue have no product, so their `PRODUCT` is `(ALL PRODUCTS)`. `AMOUNT` is only filled for revenue rows.
Quantities of different record types are different things (crates are not loaves) - filter or pivot on `RECORD_TYPE` before summing `QTY`.

In [53]:
ALL = "(ALL PRODUCTS)"
def as_facts(df, qty_col, prefix=None, rtype=None, product=None, amt_col=None):
    f = df[["DATE"] + ID_COLS].copy()
    f["PRODUCT"] = df["PRODUCT"] if product is None else product
    f["RECORD_TYPE"] = rtype if rtype else prefix + "_" + df["TYPE"]
    f["QTY"] = df[qty_col]
    f["AMOUNT"] = df[amt_col] if amt_col else np.nan
    return f

parts = [
    as_facts(brands_sheet, "BRANDS_QTY", prefix="BRANDS"),
    as_facts(hansa_sheet, "HANSA_RTN_QTY", rtype="HANSA_RETURN"),
    as_facts(mehul_sheet.assign(TYPE=mehul_sheet["TYPE"] + "_RETURN"), "MEHUL_RTN_QTY", prefix="MEHUL"),
    as_facts(crates_sheet, "CRATES_QTY", prefix="CRATES", product=ALL),
    as_facts(rev_sheet, "REV_QTY", prefix="REVENUE", product=ALL, amt_col="REV_AMT"),
]
KEYS = ["DATE"] + ID_COLS + ["PRODUCT", "RECORD_TYPE"]
combined = (pd.concat(parts, ignore_index=True)
              .groupby(KEYS, dropna=False, as_index=False)[["QTY", "AMOUNT"]].sum(min_count=1)
              .sort_values(["DATE", "CODE", "PRODUCT", "RECORD_TYPE"]).reset_index(drop=True))
combined["QTY"] = as_qty(combined["QTY"])
combined.insert(1, "MONTH", combined["DATE"].dt.strftime("%Y-%m"))
combined = combined[["DATE", "MONTH"] + ID_COLS + ["PRODUCT", "RECORD_TYPE", "QTY", "AMOUNT"]]
if len(combined) > 1_048_575:
    raise ValueError(f"Combined sheet would have {len(combined):,} rows - more than Excel allows. Narrow the date range.")
print(f"combined: {len(combined):,} rows, {combined.shape[1]} columns")

combined: 54,232 rows, 13 columns


# Accuracy checks
1. Each output is reconciled to the **raw source totals** (source total - what was cancelled = what the sheet holds).
2. Every sheet's total is the same in the datewise, precise and combined versions.
3. A cancelled invoice and the credit that cancelled it must hold the same quantity.
4. Source "Qty" columns are compared with the sum of the product columns.
The notebook stops if any check fails.

In [54]:
rows = []
def chk(name, expected, got):
    rows.append((name, float(expected), float(got), "OK" if np.isclose(expected, got) else "MISMATCH"))

# --- brands
bi, bc = b_inv[b_prods].sum().sum(), b_cn[b_prods].abs().sum().sum()
b_cancel_inv = b_inv[b_prods].sum().sum() - b_inv_kept[b_prods].sum().sum()
b_cancel_cn = b_cn_cancelled[b_prods].abs().sum().sum()
chk("brands: cancelled invoice qty = cancelling credit qty", b_cancel_inv, b_cancel_cn)
chk("brands: INVOICE qty = source - cancelled", bi - b_cancel_inv, brands_sheet.loc[brands_sheet.TYPE == "INVOICE", "BRANDS_QTY"].sum())
chk("brands: CREDIT qty = source credits" + ("" if KEEP_CANCELLED_CREDITS else " - cancelling credits"),
    bc - (0 if KEEP_CANCELLED_CREDITS else b_cancel_cn), brands_sheet.loc[brands_sheet.TYPE == "CREDIT", "BRANDS_QTY"].sum())
chk("brands: source Qty column = sum of product columns (invoices)", b_inv["QTY"].sum(), bi)
chk("brands: source Qty column = sum of product columns (credits)", b_cn["QTY"].abs().sum(), bc)
# --- crates
ci, cc = c_inv["QTY"].sum(), c_cn["QTY"].sum()
c_cancel_inv = ci - c_inv_kept["QTY"].sum()
c_cancel_cn = c_cn_cancelled["QTY"].sum()
chk("crates: cancelled invoice qty = cancelling credit qty", c_cancel_inv, c_cancel_cn)
chk("crates: INVOICE qty = source - cancelled", ci - c_cancel_inv, crates_sheet.loc[crates_sheet.TYPE == "INVOICE", "CRATES_QTY"].sum())
chk("crates: CREDIT qty = source credits" + ("" if KEEP_CANCELLED_CREDITS else " - cancelling credits"),
    cc - (0 if KEEP_CANCELLED_CREDITS else c_cancel_cn), crates_sheet.loc[crates_sheet.TYPE == "CREDIT", "CRATES_QTY"].sum())
# --- mehul / hansa
chk("mehul: sheet total = every fresh+exp cell read from the daily sheets", m_raw[val_cols].sum().sum(), mehul_sheet["MEHUL_RTN_QTY"].sum())
chk("hansa: sheet total = sum of product columns in source", h[h_prods].sum().sum(), hansa_sheet["HANSA_RTN_QTY"].sum())
chk("hansa: source Qty column = sum of product columns", h["QTY"].sum(), h[h_prods].sum().sum())
# --- revenue
chk("revenue: invoice qty", rev_inv["qty"].sum(), rev_sheet.loc[rev_sheet.TYPE == "INVOICE", "REV_QTY"].sum())
chk("revenue: invoice amount", rev_inv["amount"].sum(), rev_sheet.loc[rev_sheet.TYPE == "INVOICE", "REV_AMT"].sum())
chk("revenue: credit qty", rev_cn["qty"].sum(), rev_sheet.loc[rev_sheet.TYPE == "CREDIT", "REV_QTY"].sum())
chk("revenue: credit amount", rev_cn["amount"].sum(), rev_sheet.loc[rev_sheet.TYPE == "CREDIT", "REV_AMT"].sum())

# --- datewise vs precise vs combined
for name, dw in datewise.items():
    pr = precise_sheets[name]
    q = [c for c in dw.columns if c.endswith("_QTY")][0]
    chk(f"{name}: precise total = datewise total ({q})", dw[q].sum(), pr[q].sum())
    if name == "sales_revenue":
        chk(f"{name}: precise total = datewise total (REV_AMT)", dw["REV_AMT"].sum(), pr["REV_AMT"].sum())
for rt, g in combined.groupby("RECORD_TYPE"):
    src = {"BRANDS": (brands_sheet, "BRANDS_QTY"), "CRATES": (crates_sheet, "CRATES_QTY"), "MEHUL": (mehul_sheet, "MEHUL_RTN_QTY"),
           "REVENUE": (rev_sheet, "REV_QTY"), "HANSA": (hansa_sheet, "HANSA_RTN_QTY")}[rt.split("_")[0]]
    df_, qc = src
    if "TYPE" in df_.columns:
        want = rt.split("_", 1)[1].replace("_RETURN", "")
        tot = df_.loc[df_["TYPE"] == want, qc].sum()
    else:
        tot = df_[qc].sum()
    chk(f"combined {rt}: QTY = its sheet", tot, g["QTY"].sum())
chk("combined REVENUE amount = sales_revenue sheet", rev_sheet["REV_AMT"].sum(), combined["AMOUNT"].sum())

checks = pd.DataFrame(rows, columns=["check", "expected", "got", "status"])
print(checks.to_string(index=False))
assert (checks["status"] == "OK").all(), "A check failed - see the table above"
print("\nALL CHECKS PASSED")

                                                               check     expected          got status
               brands: cancelled invoice qty = cancelling credit qty    18925.000    18925.000     OK
                            brands: INVOICE qty = source - cancelled  1807898.000  1807898.000     OK
            brands: CREDIT qty = source credits - cancelling credits    14767.000    14767.000     OK
       brands: source Qty column = sum of product columns (invoices)  1826823.000  1826823.000     OK
        brands: source Qty column = sum of product columns (credits)    33692.000    33692.000     OK
               crates: cancelled invoice qty = cancelling credit qty     1706.000     1706.000     OK
                            crates: INVOICE qty = source - cancelled   127688.000   127688.000     OK
            crates: CREDIT qty = source credits - cancelling credits      887.000      887.000     OK
mehul: sheet total = every fresh+exp cell read from the daily sheets    23484.000 

## Write the workbooks

In [55]:
def write_workbook(path, sheets, pct_cols=()):
    path.parent.mkdir(parents=True, exist_ok=True)
    head_fill = PatternFill("solid", fgColor="1F3864")
    with pd.ExcelWriter(path, engine="openpyxl", datetime_format="yyyy-mm-dd") as xw:
        for name, df in sheets.items():
            df.to_excel(xw, sheet_name=name, index=False)
            ws = xw.sheets[name]
            for i, col in enumerate(df.columns, start=1):
                c = ws.cell(row=1, column=i)
                c.font = Font(bold=True, color="FFFFFF")
                c.fill = head_fill
                c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                sample = df[col].head(500).astype(str)
                width = max([len(str(col)) * 0.9] + [len(str(v)) for v in sample]) + 2
                ws.column_dimensions[get_column_letter(i)].width = min(max(width, 10), 45)
                if col in pct_cols:
                    for (cell,) in ws.iter_rows(min_row=2, min_col=i, max_col=i):
                        cell.number_format = "0.0%"
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
    print(f"  saved {path}: " + ", ".join(f"{n} ({len(d):,} rows x {d.shape[1]} cols)" for n, d in sheets.items()))

write_workbook(OUTPUT_DIR / "Datewise_Workbook.xlsx", datewise)
write_workbook(OUTPUT_DIR / "Precise_Workbook.xlsx", precise_sheets, pct_cols=("PCT_RETURN",))
write_workbook(OUTPUT_DIR / "Combined_Workbook.xlsx", {"combined": combined})

  saved report\test\Datewise_Workbook.xlsx: crates (5,725 rows x 10 cols), mehul_returns (6,069 rows x 11 cols), hansa_returns (4,254 rows x 10 cols), sales_revenue (6,825 rows x 11 cols), brands_perfomance (31,359 rows x 11 cols)
  saved report\test\Precise_Workbook.xlsx: crates (1,157 rows x 10 cols), mehul_returns (2,749 rows x 11 cols), hansa_returns (2,176 rows x 10 cols), sales_revenue (1,790 rows x 12 cols), brands_perfomance (7,835 rows x 11 cols)
  saved report\test\Combined_Workbook.xlsx: combined (54,232 rows x 13 cols)
